In [ ]:
from pathlib import Path
import joblib
import numpy as np
import torch

from data_utils import seed_everything
from hrv_utils import get_fold_ids, build_hrv_set
from evaluation import evaluate_hrv

DATA_PATH = Path("Data/structured_dataset_4hz.pt")
FOLD_PATH = Path("Data/subject_folds.pt")
CHECKPOINT_ROOT = Path("Trained_Models/hrv_xgboost")

HRV_HZ = 4
INPUT_SIZE = 3600 * HRV_HZ
PREDICTION_HORIZON = 3600 * HRV_HZ
SEED = 42

seed_everything(SEED)

data = torch.load(DATA_PATH, weights_only=False)
fold_data = torch.load(FOLD_PATH, weights_only=False)
af_sets, nsr_sets = fold_data["af_sets"], fold_data["nsr_sets"]

results = []

for fold_idx in range(5):
    print(f"\n===== FOLD {fold_idx} =====")

    _, val_ids, test_ids = get_fold_ids(af_sets, nsr_sets, fold_idx)

    X_val, y_val, _ = build_hrv_set(data, val_ids, INPUT_SIZE, PREDICTION_HORIZON)
    X_test, y_test, _ = build_hrv_set(data, test_ids, INPUT_SIZE, PREDICTION_HORIZON)

    model = joblib.load(CHECKPOINT_ROOT / f"fold_{fold_idx}.pkl")

    val_metrics = evaluate_hrv(model, X_val, y_val, threshold=None, target_sens=0.90)
    test_metrics = evaluate_hrv(model, X_test, y_test, threshold=val_metrics["threshold"])

    print(
        f"Sens={test_metrics['sensitivity']:.4f}, "
        f"Spec={test_metrics['specificity']:.4f}, "
        f"F1={test_metrics['f1']:.4f}, "
        f"AUROC={test_metrics['auroc']:.4f}, "
        f"AUPRC={test_metrics['auprc']:.4f}"
    )

    results.append(test_metrics)

metrics = [
    "sensitivity",
    "specificity",
    "precision",
    "f1",
    "auroc",
    "auprc"
]

print("\n===== 5-FOLD HRV RESULTS =====")
for key in metrics:
    values = np.array([r[key] for r in results])
    print(f"{key.upper()}: {values.mean():.4f} ± {values.std():.4f}")


===== FOLD 0 =====
Sens=0.9000, Spec=0.8189, F1=0.6294, AUROC=0.9257, AUPRC=0.8219

===== FOLD 1 =====
Sens=0.9778, Spec=0.5229, F1=0.4112, AUROC=0.9000, AUPRC=0.5489

===== FOLD 2 =====
Sens=0.7460, Spec=0.8438, F1=0.6483, AUROC=0.8897, AUPRC=0.7792

===== FOLD 3 =====
Sens=0.9444, Spec=0.6920, F1=0.5258, AUROC=0.9375, AUPRC=0.7635

===== FOLD 4 =====
Sens=0.8182, Spec=0.8042, F1=0.5294, AUROC=0.9171, AUPRC=0.7020

===== 5-FOLD HRV RESULTS =====
SENSITIVITY: 0.8773 ± 0.0847
SPECIFICITY: 0.7364 ± 0.1187
PRECISION: 0.4146 ± 0.1066
F1: 0.5488 ± 0.0851
AUROC: 0.9140 ± 0.0173
AUPRC: 0.7231 ± 0.0952
